In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

### Get md file from SpeedyApply github repo for US based Data Science internships

In [2]:
url = "https://raw.githubusercontent.com/speedyapply/2026-AI-College-Jobs/main/README.md"
response = requests.get(url)

with open('raw_internships.md', 'w', encoding='utf-8') as f:
    f.write(response.text)

### Extract table from raw_internships.md

In [3]:
# Helper function
import re
from io import StringIO

def extract_md_table(md_text, start_marker, end_marker):
    pattern = rf"{re.escape(start_marker)}\n(.*?)\n{re.escape(end_marker)}"
    match = re.search(pattern, md_text, flags=re.DOTALL)

    if not match:
        return None

    table_md = match.group(1).strip()

    df = pd.read_csv(
        StringIO(table_md),
        sep="|",
        engine="python"
    )

    # clean up
    df = df.dropna(axis=1, how="all")
    df.columns = df.columns.str.strip()
    df = df.iloc[1:]  # remove markdown separator row
    df = df.apply(lambda x: x.str.strip())

    # handle company link and name
    if "Company" in df.columns:
        df[["Company Link", "Company"]] = df["Company"].str.extract(
            r'href="([^"]+)".*?<strong>(.*?)</strong>'
        )

    # cleanup position link
    if "Posting" in df.columns:
        df["Posting"] = df["Posting"].str.extract(r'href="([^"]+)"')

    return df


In [4]:
# Load the markdown file
with open("raw_internships.md", "r") as f:
    md_text = f.read()

In [5]:
# Extract tables into useable dataframes
faang_df = extract_md_table(
    md_text,
    "<!-- TABLE_FAANG_START -->",
    "<!-- TABLE_FAANG_END -->"
)

quant_df = extract_md_table(
    md_text,
    "<!-- TABLE_QUANT_START -->",
    "<!-- TABLE_QUANT_END -->"
)

other_df = extract_md_table(
    md_text,
    "<!-- TABLE_START -->",
    "<!-- TABLE_END -->"
)


In [6]:
faang_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Meta,Research Scientist Intern - Mathematics - PhD,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1471311707754788,6d,https://about.meta.com
2,Meta,Research Scientist Intern - Optical System Res...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1412438827016753,6d,https://about.meta.com
3,Meta,Research Scientist Intern - Applied Vision - PhD,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1112761774275681,6d,https://about.meta.com
4,Meta,Research Scientist Intern - Novel Organic Mate...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/884653154471172,6d,https://about.meta.com
5,Meta,Research Scientist Intern - Applied Perception...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/2576688855967...,7d,https://about.meta.com


In [7]:
quant_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Citadel Securities,Quantitative Research Analyst Intern - BS/MS - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
2,Citadel Securities,Machine Learning Researcher - PhD Intern - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
3,Citadel Securities,Quantitative Researcher - PhD Intern - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers
4,Citadel,Machine Learning Researcher - PhD Intern - US,New York,$125/hr,https://www.citadel.com/careers/details/machin...,1d,https://www.citadel.com/careers
5,Citadel,Quantitative Research Analyst Intern - BS/MS - US,New York,$125/hr,https://www.citadel.com/careers/details/quanti...,1d,https://www.citadel.com/careers


In [8]:
other_df.head()

,Company,Position,Location,Posting,Age,Company Link
1,Acushnet Company,Summer 2026 Manufacturing Data Analyst Intern ...,"Dartmouth, Massachusetts, United States of Ame...",https://acushnetgolf.wd12.myworkdayjobs.com/en...,0d,https://www.acushnetholdingscorp.com/
2,Biogen,Intern - Advanced Analytics - Data Science & AI,"Cambridge, MA",https://biibhr.wd3.myworkdayjobs.com/en-US/ext...,0d,https://www.biogen.com/
3,COUNTRY Financial,Data Science Intern - DigitaLab,DigitaLab University of Illinois,https://countryfinancial.wd5.myworkdayjobs.com...,0d,https://www.countryfinancial.com/
4,Fidelity Investments,Co-op - Quantitative Analyst,"Summer St, Boston MA",https://fmr.wd1.myworkdayjobs.com/en-US/target...,0d,https://fidelity.com/
5,General Dynamics Mission Systems,Mechanical Engineering Intern - AI&R- Manassas,US VA Manassas,https://careers-gdms.icims.com/jobs/71117/mech...,0d,https://gdmissionsystems.com


### Clean the Dataframes

In [9]:
print("Faang:", faang_df.shape)
print("Quant:", quant_df.shape)
print("Other:", other_df.shape)

Faang: (66, 7)
Quant: (10, 7)
Other: (388, 6)


In [10]:
other_df.isna().sum()

Company         0
Position        0
Location        0
Posting         0
Age             0
Company Link    0
dtype: int64

In [11]:
other_df.columns

Index(['Company', 'Position', 'Location', 'Posting', 'Age', 'Company Link'], dtype='object')

In [12]:
print("Unique companies in other internships:", other_df.Company.unique().size)

Unique companies in other internships: 232


In [13]:
print("Unique companies in faang internships:", faang_df.Company.unique().size)

Unique companies in faang internships: 7


In [14]:
print("Unique companies in quant internships:", quant_df.Company.unique().size)

Unique companies in quant internships: 5


In [15]:
other_df.Company.value_counts().head(10)

Company
KLA                          10
Bosch                         5
Tencent                       5
Micron Technology             5
RippleMatch                   5
InterDigital                  5
Samsung                       5
Toyota Research Institute     5
Tatari                        5
Applied Materials             5
Name: count, dtype: int64

In [16]:
# I want to visit the links for job postings and extract more info about the interships, especially keywords in job requirement sections.
other_df.Posting.head(10)

1     https://acushnetgolf.wd12.myworkdayjobs.com/en...
2     https://biibhr.wd3.myworkdayjobs.com/en-US/ext...
3     https://countryfinancial.wd5.myworkdayjobs.com...
4     https://fmr.wd1.myworkdayjobs.com/en-US/target...
5     https://careers-gdms.icims.com/jobs/71117/mech...
6     https://jj.wd5.myworkdayjobs.com/en-US/jj/job/...
7       https://www.kargo.com/careers?gh_jid=5054569007
8     https://job-boards.greenhouse.io/legendcareers...
9     https://jobs.lever.co/MBRDNA/e214ddad-ee32-49b...
10    https://oclc.wd1.myworkdayjobs.com/en-US/oclc_...
Name: Posting, dtype: object

In [17]:
other_10 = other_df[other_df["Company"] == "Tatari"].copy()
other_10

,Company,Position,Location,Posting,Age,Company Link
40,Tatari,Data Science - Intern,"New York, New York, United States",https://job-boards.greenhouse.io/tatari/jobs/8...,4d,http://www.tatari.tv
41,Tatari,Data Science - Intern,"San Francisco, California, United States",https://job-boards.greenhouse.io/tatari/jobs/8...,4d,http://www.tatari.tv
42,Tatari,Data Science Analyst - Intern,"Los Angeles, California, United States",https://job-boards.greenhouse.io/tatari/jobs/8...,4d,http://www.tatari.tv
43,Tatari,Data Science Analyst - Intern,"San Francisco, California, United States",https://job-boards.greenhouse.io/tatari/jobs/8...,4d,http://www.tatari.tv
44,Tatari,Data Science Analyst - Intern,"New York, New York, United States",https://job-boards.greenhouse.io/tatari/jobs/8...,4d,http://www.tatari.tv


In [18]:
link = other_10["Posting"].head(1)
link = link.values[0]
link

'https://job-boards.greenhouse.io/tatari/jobs/8432059002'

In [19]:
url = "https://job-boards.greenhouse.io/tatari/jobs/8432060002"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

# Main job content container
description = soup.find("div", class_="job__description")

print(description.get_text(separator="\n",strip=True))
str(description)

Tatari is on a mission to revolutionize TV advertising. Founded in 2016 to help transform the antiquated world of TV advertising through the intelligent application of AI and machine learning, Tatari helps some of the world’s fastest growing brands including
Chime, Calm, Tecovas, Manscaped, Saatva, and Liquid I.V
., reach their customers using linear and streaming TV ads. Our platform combines sophisticated media buying with proprietary analytics to turn TV advertising into an automated, digital-like experience, enabling businesses of any size to advertise on TV.
That approach has earned Tatari broad industry recognition, including being named
Best CTV AdTech Platform
in the 8th annual
MarTech Breakthrough Awards
, as well as honors from
Digiday
(
Best Connected TV Platform)
,
AdExchanger
(
Most Innovative TV Advertising Technology
), and
Business Insider
(
Hottest AdTech Companies)
. Tatari has also been recognized as the
Best Place to Work
by
Inc. Magazine
. Backed by an executive te

'<div class="job__description body"><div><p>Tatari is on a mission to revolutionize TV advertising. Founded in 2016 to help transform the antiquated world of TV advertising through the intelligent application of AI and machine learning, Tatari helps some of the world’s fastest growing brands including <em>Chime, Calm, Tecovas, Manscaped, Saatva, and Liquid I.V</em>., reach their customers using linear and streaming TV ads. Our platform combines sophisticated media buying with proprietary analytics to turn TV advertising into an automated, digital-like experience, enabling businesses of any size to advertise on TV.</p>\n<p>That approach has earned Tatari broad industry recognition, including being named\xa0<a href="https://www.tatari.tv/insights/tatari-named-best-ctv-adtech-platform-in-2025-martech-breakthrough-awards">Best CTV AdTech Platform</a> in the 8th annual <em>MarTech Breakthrough Awards</em>, as well as honors from <em>Digiday</em> (<a href="https://www.tatari.tv/insights/tata

In [20]:
other_df['raw_description'] = None
other_df['description_text'] = None
other_df.head()

,Company,Position,Location,Posting,Age,Company Link,raw_description,description_text
1,Acushnet Company,Summer 2026 Manufacturing Data Analyst Intern ...,"Dartmouth, Massachusetts, United States of Ame...",https://acushnetgolf.wd12.myworkdayjobs.com/en...,0d,https://www.acushnetholdingscorp.com/,None,None
2,Biogen,Intern - Advanced Analytics - Data Science & AI,"Cambridge, MA",https://biibhr.wd3.myworkdayjobs.com/en-US/ext...,0d,https://www.biogen.com/,None,None
3,COUNTRY Financial,Data Science Intern - DigitaLab,DigitaLab University of Illinois,https://countryfinancial.wd5.myworkdayjobs.com...,0d,https://www.countryfinancial.com/,None,None
4,Fidelity Investments,Co-op - Quantitative Analyst,"Summer St, Boston MA",https://fmr.wd1.myworkdayjobs.com/en-US/target...,0d,https://fidelity.com/,None,None
5,General Dynamics Mission Systems,Mechanical Engineering Intern - AI&R- Manassas,US VA Manassas,https://careers-gdms.icims.com/jobs/71117/mech...,0d,https://gdmissionsystems.com,None,None


In [21]:

# Scraper helper functions — one extractor per job board

import numpy as np
import json
from urllib.parse import urlparse

try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

def _get(url, timeout=15):
    return requests.get(url, headers=_HEADERS, timeout=timeout)

def _to_clean(html_str):
    return BeautifulSoup(html_str, "html.parser").get_text(separator="\n", strip=True)

def _try_json_ld(soup):
    """Extract description from JSON-LD JobPosting schema markup."""
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
            if isinstance(data, list):
                data = data[0] if data else {}
            if data.get("@type") == "JobPosting":
                desc = data.get("description", "")
                if desc:
                    return desc, _to_clean(desc)
        except Exception:
            pass
    return np.nan, np.nan


def _extract_greenhouse(url):
    soup = BeautifulSoup(_get(url).text, "html.parser")
    div = soup.find("div", class_="job__description")
    if div:
        return str(div), div.get_text(separator="\n", strip=True)
    return _try_json_ld(soup)


def _extract_ashby(url):
    """Use Ashby's public posting API."""
    parts = urlparse(url).path.strip("/").split("/")
    if len(parts) >= 2:
        company, job_id = parts[0], parts[1]
        api = f"https://api.ashbyhq.com/posting-api/job-board/{company}/postings/{job_id}"
        data = _get(api).json()
        raw = data.get("descriptionHtml", "")
        if raw:
            return raw, _to_clean(raw)
    return np.nan, np.nan


def _extract_workday(url):
    """Use Workday's internal CXS API, fall back to JSON-LD."""
    parsed = urlparse(url)
    host   = parsed.hostname.split(".")          # e.g. ['caci', 'wd1', 'myworkdayjobs', 'com']
    company, wd = host[0], host[1]
    path   = [p for p in parsed.path.split("/") if p]  # ['en-US', 'board', 'job', ..., 'slug_ID']
    if len(path) >= 2:
        board = path[1]
        m = re.search(r'_([A-Za-z0-9]+)$', path[-1])
        if m:
            job_id = m.group(1)
            api = f"https://{company}.{wd}.myworkdayjobs.com/wday/cxs/{company}/{board}/jobs/{job_id}"
            try:
                data = _get(api).json()
                desc = data.get("jobPostingInfo", {}).get("jobDescription", "")
                if desc:
                    return desc, _to_clean(desc)
            except Exception:
                pass
    # Fallback: JSON-LD on the rendered page
    try:
        soup = BeautifulSoup(_get(url).text, "html.parser")
        return _try_json_ld(soup)
    except Exception:
        return np.nan, np.nan


def _extract_lever(url):
    clean_url = re.sub(r'/apply/?$', '', url)   # strip trailing /apply
    soup = BeautifulSoup(_get(clean_url).text, "html.parser")
    div = (
        soup.find("div", {"data-qa": "job-description"})
        or soup.find("div", class_="section-wrapper")
        or soup.find("div", class_="content")
    )
    if div:
        return str(div), div.get_text(separator="\n", strip=True)
    return _try_json_ld(soup)


def _extract_smartrecruiters(url):
    """Use SmartRecruiters public API, fall back to page scrape."""
    parsed = urlparse(url)
    parts  = parsed.path.strip("/").split("/")   # [CompanyName, jobId-slug...]
    if len(parts) >= 2:
        company = parts[0]
        job_id  = parts[1].split("-")[0]
        api = f"https://api.smartrecruiters.com/v1/companies/{company}/postings/{job_id}"
        try:
            data = _get(api).json()
            sections = data.get("jobAd", {}).get("sections", {})
            texts = [
                sections.get(k, {}).get("text", "")
                for k in ("jobDescription", "qualifications", "additionalInformation")
            ]
            combined = "\n\n".join(t for t in texts if t)
            if combined:
                return combined, _to_clean(combined)
        except Exception:
            pass
    try:
        soup = BeautifulSoup(_get(url).text, "html.parser")
        div = soup.find("div", class_=re.compile(r"job-description", re.I))
        if div:
            return str(div), div.get_text(separator="\n", strip=True)
        return _try_json_ld(soup)
    except Exception:
        return np.nan, np.nan


def _extract_generic(url):
    soup = BeautifulSoup(_get(url).text, "html.parser")
    result = _try_json_ld(soup)
    if pd.notna(result[0]):
        return result
    for tag, attrs in [
        ("div", {"id": "job-description"}),
        ("div", {"class": "job-description"}),
        ("div", {"class": "jobDescription"}),
        ("div", {"class": "description"}),
        ("div", {"class": "posting-description"}),
        ("section", {"class": re.compile(r"description", re.I)}),
    ]:
        div = soup.find(tag, attrs)
        if div:
            return str(div), div.get_text(separator="\n", strip=True)
    return np.nan, np.nan


def fetch_description(url):
    """Route to the right extractor. Returns (raw_html, clean_text) or (NaN, NaN)."""
    if not url or (isinstance(url, float) and np.isnan(url)):
        return np.nan, np.nan
    try:
        domain = urlparse(url).hostname or ""
        if "greenhouse.io" in domain:
            raw, text = _extract_greenhouse(url)
        elif "ashbyhq.com" in domain:
            raw, text = _extract_ashby(url)
        elif "myworkdayjobs.com" in domain:
            raw, text = _extract_workday(url)
        elif "lever.co" in domain:
            raw, text = _extract_lever(url)
        elif "smartrecruiters.com" in domain:
            raw, text = _extract_smartrecruiters(url)
        else:
            raw, text = _extract_generic(url)
        return (
            raw  if pd.notna(raw)  else np.nan,
            text if pd.notna(text) else np.nan,
        )
    except Exception:
        return np.nan, np.nan


In [22]:

# Scrape job descriptions for all rows in other_df

DELAY = 0.5  # seconds between requests

raw_descs, clean_descs = [], []

for i, url in enumerate(tqdm(other_df["Posting"], total=len(other_df), desc="Fetching descriptions")):
    raw, clean = fetch_description(url)
    raw_descs.append(raw)
    clean_descs.append(clean)

    if (i + 1) % 50 == 0:
        n_ok = sum(1 for x in clean_descs if pd.notna(x))
        print(f"  [{i+1:3d}/{len(other_df)}] extracted so far: {n_ok}")

    time.sleep(DELAY)

other_df["raw_description"]  = raw_descs
other_df["description_text"] = clean_descs

n_ok = other_df["description_text"].notna().sum()
print(f"\nDone! Extracted {n_ok}/{len(other_df)} descriptions ({100*n_ok/len(other_df):.1f}%)")
other_df[["Company", "Position", "description_text"]].head(10)


Fetching descriptions:   0%|          | 0/388 [00:00<?, ?it/s]

  [ 50/388] extracted so far: 39
  [100/388] extracted so far: 71
  [150/388] extracted so far: 112
  [200/388] extracted so far: 155
  [250/388] extracted so far: 196
  [300/388] extracted so far: 241
  [350/388] extracted so far: 266

Done! Extracted 290/388 descriptions (74.7%)


,Company,Position,description_text
1,Acushnet Company,Summer 2026 Manufacturing Data Analyst Intern ...,Where Performance Meets Purpose Join a team th...
2,Biogen,Intern - Advanced Analytics - Data Science & AI,About the Role: This application is for a 12-w...
3,COUNTRY Financial,Data Science Intern - DigitaLab,Experience more with a career at COUNTRY Finan...
4,Fidelity Investments,Co-op - Quantitative Analyst,Job Description: Quantitative Analyst Descript...
5,General Dynamics Mission Systems,Mechanical Engineering Intern - AI&R- Manassas,NaN
6,Johnson & Johnson,Ethics and AI Ops Intern,"At Johnson & Johnson,â¯we believe health is e..."
7,Kargo,Intern - Data Analyst,NaN
8,Legend Biotech,AI/ML Intern,NaN
9,Mercedes-Benz R&D,ML Ops Intern,At Mercedes-Benz Research & Development North ...
10,OCLC,Data Science Summer Intern,Together we make breakthroughs possible. At OC...


In [23]:
other_df.isna().sum()

Company              0
Position             0
Location             0
Posting              0
Age                  0
Company Link         0
raw_description     98
description_text    98
dtype: int64

In [24]:
other_df.shape

(388, 8)

In [25]:
other_df["description_text"][10]

'Together we make breakthroughs possible. At OCLC, we build technology with a purpose: to connect libraries and make knowledge accessible worldwide, because we believe that what is known must be shared. Our teams work with complex global datasets, AI and machine learning, hybrid cloud solutions, and other technologies that connect people and organizations to the information they need. We value the power of unique perspectives and experiences to unlock innovation. At OCLC, your ideas matter, whether you have two years of experience or 20. Youâ\x80\x99ll learn, create, and problem-solve with technologists, product developers, librarians, researchers, marketing pros, and support teams around the world. Why join OCLC? OCLC is consistently recognized as a best place to work by several independent programs. We recognize and reward people and results with a comprehensive Total Rewards package. This means competitive compensation that reflects your unique contributionsâ\x80\x94performance, exp